# YOLO - detekcja obiektów w jednym przebiegu sieci

Notatnik demonstracyjny. Analiza Danych Obrazowych i Multimedialnych (ADOM), semestr 2026L, TEMAT 13.

Zespół 12: Jan Milczarek, Julia Syzdół, Nazar Shcherbyna, Aleksandra Shuliakouskaya.

Artykuł źródłowy: *You Only Look Once* (Redmon i in., 2016, [arXiv:1506.02640](https://arxiv.org/abs/1506.02640)).
Implementacja: [Ultralytics](https://docs.ultralytics.com).

Uruchamiaj komórki po kolei (Shift+Enter). Sekcje 1-3 to teoria i instalacja, sekcje 4-5 uruchamiają YOLO na żywo, a sekcje 6-7 pokazują wyniki naszych eksperymentów (dane są wpisane w notatniku, nic nie trzeba trenować ani pobierać). Działa w Google Colab i lokalnie, na GPU lub CPU.

## 1. Czym jest YOLO

YOLO (You Only Look Once) to sieć neuronowa do detekcji obiektów, opisana w 2016 roku przez Josepha Redmona i współpracowników. Detekcja to jednoczesne znalezienie obiektów (ramki) i rozpoznanie ich klas na obrazie.

Główna idea: sieć ogląda obraz tylko raz. Starsze metody (np. R-CNN) najpierw proponowały setki regionów, a potem każdy klasyfikowały osobno, przez co były wolne. YOLO robi to w jednym przebiegu:

- obraz jest dzielony na siatkę S x S,
- każda komórka przewiduje B ramek (położenie, rozmiar, pewność) oraz C klas,
- wyjście ma rozmiar S x S x (B * 5 + C).

Dzięki temu model widzi cały obraz naraz i działa w czasie rzeczywistym.

| | Dwuetapowe (R-CNN) | Jednoetapowe (YOLO) |
|---|---|---|
| Sposób | regiony, potem klasyfikacja | jeden przebieg sieci |
| Szybkość | wolniej | szybko, real-time |
| Zastosowanie | offline, maks. dokładność | wideo, kamery, edge |

## 2. Implementacja - Ultralytics

Korzystamy z trzech generacji udostępnianych przez firmę Ultralytics: YOLOv8 (2023), YOLO11 (2024) i YOLO26 (2025). Framework jest wysokopoziomowy - model ładuje się jedną linijką, a detekcja to wywołanie predict().

Każdy model występuje w rozmiarach: n (nano), s (small), m (medium), l (large), x (extra-large). Im większy, tym dokładniejszy, ale wolniejszy. Nazwa łączy wersję i rozmiar, np. yolo11n to wersja 11 w rozmiarze nano.

## 3. Instalacja

Instalujemy ultralytics (wraz z PyTorch) oraz pandas do tabel z wynikami. W Colab środowisko z GPU jest gotowe od razu; lokalnie do GPU potrzebne są sterowniki CUDA, ale na CPU też działa.

In [ ]:
%pip install -q ultralytics pandas

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from IPython.display import display

import ultralytics
from ultralytics import YOLO
from ultralytics.utils import ASSETS

mpl.rcParams.update({
    "figure.dpi": 120,
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
    "axes.titlepad": 12,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "axes.axisbelow": True,
    "grid.color": "#ececec",
})
NAVY, ORANGE, GRAY = "#1f2a44", "#e8743b", "#9aa3b2"

print("Ultralytics", ultralytics.__version__)

## 4. YOLO w akcji

Detekcja to dwa kroki: wczytaj model i wywołaj predykcję. Używamy yolo11n (najmniejszy, najszybszy) na zdjęciach dołączonych do pakietu, więc komórka działa nawet bez internetu. Parametr conf=0.35 to próg pewności - pokazujemy tylko detekcje pewne w co najmniej 35 procentach.

In [ ]:
model = YOLO("yolo11n.pt")

images = [ASSETS / "bus.jpg", ASSETS / "zidane.jpg"]
results = model.predict(images, conf=0.35, verbose=False)

fig, axes = plt.subplots(len(images), 2, figsize=(11, 4.4 * len(images)))
for row, r in zip(axes, results):
    row[0].imshow(r.orig_img[..., ::-1])
    row[0].set_title("wejscie")
    row[1].imshow(r.plot()[..., ::-1])
    row[1].set_title(f"YOLO11n: {len(r.boxes)} obiektow")
    for a in row:
        a.axis("off")
plt.tight_layout()
plt.show()

for r in results:
    names = [model.names[int(c)] for c in r.boxes.cls]
    counts = pd.Series(names).value_counts()
    print(Path(r.path).name, "->", ", ".join(f"{k} x{v}" for k, v in counts.items()))

Wynik (r.boxes) zawiera położenia ramek, klasy i pewności. Z tego można liczyć obiekty, śledzić je w wideo czy zliczać ludzi w kadrze. Jedna detekcja zajmuje od kilkunastu do kilkudziesięciu milisekund, dlatego YOLO nadaje się do kamer i wideo na żywo.

## 5. Szybkość a dokładność

Główny wątek pracy z 2016 roku: detekcja jednoetapowa to wymiana między szybkością a dokładnością. Pokazujemy to na żywo - ten sam obraz przez trzy rozmiary YOLO11 (n, s, m). Większy model działa dłużej.

Zmierzony czas zależy od sprzętu, więc patrz na niego względnie (n vs s vs m). Powtarzalne pomiary na całym zbiorze są w sekcji 6.

In [ ]:
sizes = ["yolo11n.pt", "yolo11s.pt", "yolo11m.pt"]
img = ASSETS / "bus.jpg"

rows = []
for name in sizes:
    m = YOLO(name)
    m.predict(img, verbose=False)                   # rozgrzewka
    t0 = time.perf_counter()
    r = m.predict(img, verbose=False)[0]
    dt = (time.perf_counter() - t0) * 1000
    rows.append((name.replace(".pt", ""), round(dt, 1), len(r.boxes)))

lat = pd.DataFrame(rows, columns=["model", "ms", "obiekty"])

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(lat.model, lat.ms, color=[GRAY, NAVY, ORANGE], width=0.6)
ax.bar_label(bars, fmt="%.0f ms", padding=3)
ax.set_ylabel("czas inferencji [ms]")
ax.set_ylim(0, lat.ms.max() * 1.18)
ax.set_title("Ten sam obraz, trzy rozmiary modelu")
plt.tight_layout()
plt.show()

display(lat)

**Czas rzeczywisty i śledzenie.** Ponieważ jeden przebieg trwa kilkadziesiąt ms, YOLO przetwarza wideo na żywo. Tryb track dodatkowo nadaje obiektom identyfikatory (ten sam obiekt zachowuje to samo ID między klatkami):

```python
model.track(source=0, show=True)   # 0 = kamera, albo sciezka do pliku lub URL wideo
```

W Google Colab podgląd okna (show=True) nie działa - uruchom śledzenie lokalnie albo zapisz wynik do pliku (save=True).

## 6. Nasze eksperymenty

Badaliśmy kompromis dokładność-szybkość oraz tezę, że nowsze lub większe nie znaczy zawsze lepsze poza COCO. Modele wytrenowane na COCO ewaluowaliśmy bez dotrenowania (zero-shot) na innym zbiorze - Pascal VOC (test2007, 4952 obrazy). Działa to, bo 20 klas VOC zawiera się w 80 klasach COCO. Dla kontroli trudności zdjęć powtórzyliśmy pomiary na COCO-20 (val2017 ograniczone do tych samych 20 klas, trudniejsze zdjęcia).

| Eksperyment | Co zmieniamy | Co stałe |
|---|---|---|
| A - wersja | yolov8n, yolo11n, yolo26n | rozmiar n, imgsz 640 |
| B - rozmiar | yolo11 n, s, m | wersja 11, imgsz 640 |
| C - rozdzielczość | imgsz 320, 640, 960 | yolo11n |
| D - próg NMS | iou 0.45, 0.6, 0.7 | yolo11n, imgsz 640 |

Miary: mAP@50 i mAP@50-95 (dokładność), ms/obraz (czas mierzony na całym zbiorze i dzielony przez liczbę obrazów), liczba parametrów i GFLOPs. Sprzęt: CPU (Intel i5-8300H). Dane poniżej są wpisane wprost z naszych pomiarów, więc wykresy odtwarzają się bez uruchamiania eksperymentów.

In [ ]:
# Wyniki pomiarow (zero-shot, CPU) - wpisane na stale, by notatnik byl samowystarczalny
VOC = pd.DataFrame([
    # model      imgsz  iou  params  GFLOPs  mAP50   mAP50-95  ms/obraz
    ("yolov8n",  640, 0.60,  3.16,  8.86, 0.8077, 0.5979,  57.51),
    ("yolo11n",  640, 0.60,  2.62,  6.61, 0.8227, 0.6219,  54.66),
    ("yolo26n",  640, 0.60,  2.57,  6.12, 0.8172, 0.6309,  58.50),
    ("yolo11s",  640, 0.60,  9.46, 21.72, 0.8614, 0.6772, 145.34),
    ("yolo11m",  640, 0.60, 20.11, 68.53, 0.8707, 0.7041, 394.17),
    ("yolo11n",  320, 0.60,  2.62,  1.65, 0.7700, 0.5658,  21.21),
    ("yolo11n",  960, 0.60,  2.62, 14.88, 0.7692, 0.5054, 147.99),
    ("yolo11n",  640, 0.45,  2.62,  6.61, 0.8205, 0.6153,  65.07),
    ("yolo11n",  640, 0.70,  2.62,  6.61, 0.8198, 0.6244,  65.77),
], columns=["model", "imgsz", "iou", "params_M", "GFLOPs", "mAP50", "mAP50_95", "ms_img"])

# mAP@50-95 na trzech poziomach trudnosci (imgsz 640, iou 0.6); COCO-80 = oficjalne docs
DIFF = pd.DataFrame([
    ("yolov8n", 0.5979, 0.4333, 0.373),
    ("yolo11n", 0.6219, 0.4575, 0.395),
    ("yolo26n", 0.6309, 0.4634, 0.409),
    ("yolo11s", 0.6772, 0.5271, 0.470),
    ("yolo11m", 0.7041, 0.5704, 0.515),
], columns=["model", "VOC20", "COCO20", "COCO80"])

base = VOC[(VOC.imgsz == 640) & (VOC.iou == 0.60)]
display(base[["model", "params_M", "GFLOPs", "mAP50", "mAP50_95", "ms_img"]]
        .round(3).reset_index(drop=True))

### A - wersja

Trzy generacje w rozmiarze nano: v8n (2023), 11n (2024), 26n (2025).

In [ ]:
ver = ["yolov8n", "yolo11n", "yolo26n"]
A = VOC[(VOC.model.isin(ver)) & (VOC.imgsz == 640) & (VOC.iou == 0.60)].set_index("model").reindex(ver)

x = np.arange(len(ver))
w = 0.38
fig, ax = plt.subplots(figsize=(7.2, 4.4))
b1 = ax.bar(x - w / 2, A.mAP50,    w, color=NAVY,   label="mAP@50")
b2 = ax.bar(x + w / 2, A.mAP50_95, w, color=ORANGE, label="mAP@50-95")
ax.bar_label(b1, fmt="%.3f", padding=2, fontsize=9)
ax.bar_label(b2, fmt="%.3f", padding=2, fontsize=9)
ax.set_xticks(x)
ax.set_xticklabels(ver)
ax.set_ylim(0, 1.12)
ax.set_ylabel("mAP")
ax.legend(frameon=False, ncol=2, loc="upper center")
ax.set_title("Eksp. A: wersja YOLO (rozmiar n, VOC)")
plt.tight_layout()
plt.show()

print("mAP@50: najlepszy yolo11n", round(A.loc["yolo11n", "mAP50"], 4),
      "- nowszy yolo26n nie wygrywa", round(A.loc["yolo26n", "mAP50"], 4))
print("mAP@50-95: yolo26n prowadzi", round(A.loc["yolo26n", "mAP50_95"], 4),
      "i jest najlzejszy", round(A.loc["yolo26n", "params_M"], 2), "M params, bez NMS")

### B - rozmiar modelu

Ta sama wersja (11) w trzech rozmiarach. Słupki (lewa oś) to dokładność, linia (prawa oś) to czas.

In [ ]:
sz = ["yolo11n", "yolo11s", "yolo11m"]
B = VOC[(VOC.model.isin(sz)) & (VOC.imgsz == 640) & (VOC.iou == 0.60)].set_index("model").reindex(sz)

fig, ax1 = plt.subplots(figsize=(7.2, 4.4))
bars = ax1.bar(sz, B.mAP50_95, color=NAVY, width=0.55)
ax1.bar_label(bars, fmt="%.3f", padding=3)
ax1.set_ylabel("mAP@50-95", color=NAVY)
ax1.set_ylim(0, 0.85)

ax2 = ax1.twinx()
ax2.grid(False)
ax2.plot(sz, B.ms_img, "o-", color=ORANGE, lw=2.2)
ax2.set_ylabel("ms / obraz (CPU)", color=ORANGE)
ax2.set_ylim(0, B.ms_img.max() * 1.22)
for s, v in zip(sz, B.ms_img):
    ax2.annotate(f"{v:.0f} ms", (s, v), color=ORANGE, xytext=(8, 4),
                 textcoords="offset points", ha="left", fontsize=9)
ax1.set_title("Eksp. B: wiekszy = dokladniejszy, ale wolniejszy (VOC)")
plt.tight_layout()
plt.show()

t_ns = B.loc["yolo11s", "ms_img"] / B.loc["yolo11n", "ms_img"]
t_nm = B.loc["yolo11m", "ms_img"] / B.loc["yolo11n", "ms_img"]
print(f"n -> s: czas razy {t_ns:.1f}.  n -> m: czas razy {t_nm:.1f}. Przyrosty dokladnosci maleja.")

### C - rozdzielczość wejścia

Modele trenowano przy około 640 px. Sprawdzamy 320 i 960.

In [ ]:
C = VOC[(VOC.model == "yolo11n") & (VOC.iou == 0.60)].sort_values("imgsz")

fig, ax = plt.subplots(figsize=(7.2, 4.4))
ax.plot(C.imgsz, C.mAP50_95, "o-", color=NAVY, lw=2.4, ms=8, zorder=4)
peak = C.loc[C.mAP50_95.idxmax()]
ax.scatter([peak.imgsz], [peak.mAP50_95], s=240, facecolors="none",
           edgecolors=ORANGE, lw=2.5, zorder=5)
for _, r in C.iterrows():
    dy = 12 if r.imgsz != peak.imgsz else 16
    ax.annotate(f"{r.mAP50_95:.3f}", (r.imgsz, r.mAP50_95), xytext=(0, dy),
                textcoords="offset points", ha="center", fontsize=10)
ax.annotate("optimum (640)", (peak.imgsz, peak.mAP50_95), color=ORANGE,
            xytext=(0, -22), textcoords="offset points", ha="center",
            fontsize=10, fontweight="bold")
ax.set_xticks(C.imgsz)
ax.set_xlabel("imgsz [px]")
ax.set_ylabel("mAP@50-95")
ax.set_ylim(C.mAP50_95.min() - 0.04, C.mAP50_95.max() + 0.05)
ax.set_title("Eksp. C: wieksza rozdzielczosc nie znaczy lepiej (yolo11n, VOC)")
plt.tight_layout()
plt.show()

print("960 px wypada gorzej niz 320 px (0.505 < 0.566) mimo okolo 9x wiekszego kosztu.")
print("Upscaling zmienia skale obiektow wzgledem treningu i psuje detekcje.")

### D - próg IoU w NMS

NMS usuwa zduplikowane ramki wskazujące ten sam obiekt. Próg iou decyduje, jak mocno ramki muszą się pokrywać, by jedną odrzucić.

In [ ]:
D = pd.DataFrame([
    (0.45, 0.8205, 0.6153, 0.793, 0.761),
    (0.60, 0.8227, 0.6219, 0.790, 0.762),
    (0.70, 0.8198, 0.6244, 0.788, 0.760),
], columns=["iou", "mAP50", "mAP50_95", "precision", "recall"])

print("Wplyw progu NMS jest marginalny (rzad 0.01) - yolo11n, imgsz 640, VOC:")
display(D)

### Trudność zbioru

Te same modele na trzech poziomach trudności: VOC-20 (łatwiej), COCO-20 (te same klasy, trudniejsze zdjęcia), COCO-80 (pełne COCO, oficjalne liczby). To najmocniejszy argument za tezą prowadzącego.

In [ ]:
order = DIFF.model.tolist()
x = np.arange(len(order))
w = 0.26
fig, ax = plt.subplots(figsize=(8.6, 4.6))
ax.bar(x - w, DIFF.VOC20 * 100,  w, color=GRAY,   label="VOC-20 (latwiej)")
ax.bar(x,     DIFF.COCO20 * 100, w, color=NAVY,   label="COCO-20")
ax.bar(x + w, DIFF.COCO80 * 100, w, color=ORANGE, label="COCO-80 (trudniej)")
ax.set_xticks(x)
ax.set_xticklabels(order)
ax.set_ylabel("mAP@50-95 [%]")
ax.set_ylim(0, 92)
ax.legend(frameon=False, ncol=3, loc="upper center")
ax.set_title("Im trudniejszy zbior, tym wiekszy zysk z lepszego modelu")
plt.tight_layout()
plt.show()

def gain(col):
    n = DIFF.loc[DIFF.model == "yolo11n", col].values[0]
    m = DIFF.loc[DIFF.model == "yolo11m", col].values[0]
    return (m / n - 1) * 100

print(f"Zysk z rozmiaru (n -> m): VOC-20 {gain('VOC20'):.0f}%, "
      f"COCO-20 {gain('COCO20'):.0f}%, COCO-80 {gain('COCO80'):.0f}%. Rosnie z trudnoscia.")

### Kompromis dokładność - czas

Każdy punkt to jedna konfiguracja na VOC. Pomarańczowa linia to front Pareto - najlepsza dokładność osiągalna przy danym czasie. Podpisane są tylko punkty na froncie (reszta to konfiguracje zdominowane).

In [ ]:
P = VOC.sort_values("ms_img")
front, best = [], -1.0
for idx, r in P.iterrows():
    if r.mAP50_95 > best:
        front.append(idx)
        best = r.mAP50_95
is_front = VOC.index.isin(front)

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(VOC.loc[~is_front, "ms_img"], VOC.loc[~is_front, "mAP50_95"],
           s=65, color=GRAY, zorder=3, label="zdominowane")
fr = VOC.loc[is_front].sort_values("ms_img")
ax.plot(fr.ms_img, fr.mAP50_95, "--", color=ORANGE, lw=1.5, zorder=2)
ax.scatter(fr.ms_img, fr.mAP50_95, s=95, color=ORANGE, zorder=4, label="front Pareto")
for _, r in fr.iterrows():
    ax.annotate(f"{r.model} @{int(r.imgsz)}", (r.ms_img, r.mAP50_95),
                xytext=(9, -10), textcoords="offset points", fontsize=9, color="#b9531f")
ax.set_xlim(0, VOC.ms_img.max() * 1.12)
ax.set_xlabel("ms / obraz (CPU), mniej = szybciej")
ax.set_ylabel("mAP@50-95, wiecej = dokladniej")
ax.set_title("Kompromis dokladnosc - szybkosc (VOC)")
ax.legend(frameon=False, loc="lower right")
plt.tight_layout()
plt.show()

## 7. Wnioski

1. Nowsze nie znaczy zawsze lepsze poza COCO (A). Na mAP@50 starszy yolo11n (2024) bije najnowszy yolo26n (2025) na obu zbiorach (VOC i COCO-20). yolo26n wygrywa dopiero na mAP@50-95, jest za to najlżejszy i nie używa NMS. Różnice między generacjami nano są małe (około 3 pp).

2. Większy model = dokładniejszy, ale wolniejszy (B). mAP rośnie n, s, m, ale przyrosty maleją, a czas mocno rośnie (yolo11m to około 7x czas yolo11n na CPU). Najlepszy stosunek jakości do czasu daje przejście n na s.

3. Większa rozdzielczość nie znaczy lepiej (C). Zależność jest niemonotoniczna - optimum przy 640, a 960 wypada gorzej niż 320 mimo około 9x większego kosztu. Nie warto skalować wejścia w górę względem treningu.

4. Próg NMS to drobne strojenie (D). Wpływ iou na mAP jest marginalny (około 0.01); 0.6 to rozsądny domyślny wybór.

5. Im trudniejszy zbiór, tym większy zysk z lepszego modelu. Na osi VOC-20, COCO-20, COCO-80 przyrost z rozmiaru rośnie: 13%, 25%, 30%. Przewaga zmierzona na pełnym COCO nie przenosi się wprost na inny zbiór, dlatego warto testować na własnych danych.

Rekomendacja: do czasu rzeczywistego yolo11n lub yolo11s przy imgsz 640; do maksymalnej dokładności offline yolo11m; model i rozdzielczość dobieraj do własnego zbioru, a nie do rankingów na COCO.